In [1]:
import sys
sys.path.append("/Users/chris/Documents/higs-vision")
import numpy as np
import time
import json

from src.data import load_config, load_processed
from src.evaluate import compute_metrics, compute_ece, save_metrics, print_comparison_table, load_metrics

config = load_config("/Users/chris/Documents/higs-vision/config.yaml")
X_train, y_train, X_val, y_val, X_test, y_test, scaler = load_processed(config)

Loaded processed data from /Users/chris/Documents/higs-vision/data/processed/
  Train: (700000, 28), Val: (150000, 28), Test: (150000, 28)


In [2]:
from sklearn.linear_model import LogisticRegression

# Train
start = time.time()
lr = LogisticRegression(
    C=1.0, max_iter=1000, random_state=config["seeds"]["data_split"], n_jobs=-1
)
lr.fit(X_train, y_train)
lr_time = time.time() - start

# Predict
lr_proba = lr.predict_proba(X_test)[:, 1]
lr_metrics = compute_metrics(y_test, lr_proba)
lr_metrics["training_time_seconds"] = round(lr_time, 1)

# Save
save_metrics(lr_metrics, "results/metrics/logistic_regression.json")
print(f"Logistic Regression trained in {lr_time:.1f}s")
print(f"Test ROC-AUC: {lr_metrics['roc_auc']:.4f}")

Metrics saved to results/metrics/logistic_regression.json
Logistic Regression trained in 3.0s
Test ROC-AUC: 0.6850


In [3]:
from sklearn.ensemble import RandomForestClassifier

start = time.time()
rf = RandomForestClassifier(
    n_estimators=config["baselines"]["random_forest"]["n_estimators"],
    max_depth=None,
    random_state=config["seeds"]["random_forest"],
    n_jobs=-1
)
rf.fit(X_train, y_train)
rf_time = time.time() - start

rf_proba = rf.predict_proba(X_test)[:, 1]
rf_metrics = compute_metrics(y_test, rf_proba)
rf_metrics["training_time_seconds"] = round(rf_time, 1)

save_metrics(rf_metrics, "results/metrics/random_forest.json")
print(f"Random Forest trained in {rf_time:.1f}s")
print(f"Test ROC-AUC: {rf_metrics['roc_auc']:.4f}")

Metrics saved to results/metrics/random_forest.json
Random Forest trained in 343.7s
Test ROC-AUC: 0.8201


In [4]:
from xgboost import XGBClassifier

start = time.time()
xgb = XGBClassifier(
    n_estimators=config["baselines"]["xgboost"]["n_estimators"],
    learning_rate=config["baselines"]["xgboost"]["learning_rate"],
    max_depth=config["baselines"]["xgboost"]["max_depth"],
    early_stopping_rounds=config["baselines"]["xgboost"]["early_stopping_rounds"],
    random_state=config["seeds"]["xgboost"],
    eval_metric="auc",
    n_jobs=-1
)
xgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=10
)
xgb_time = time.time() - start

xgb_proba = xgb.predict_proba(X_test)[:, 1]
xgb_metrics = compute_metrics(y_test, xgb_proba)
xgb_metrics["training_time_seconds"] = round(xgb_time, 1)

save_metrics(xgb_metrics, "results/metrics/xgboost.json")
print(f"XGBoost trained in {xgb_time:.1f}s")
print(f"Test ROC-AUC: {xgb_metrics['roc_auc']:.4f}")

[0]	validation_0-auc:0.74192
[10]	validation_0-auc:0.77186
[20]	validation_0-auc:0.78681
[30]	validation_0-auc:0.79479
[40]	validation_0-auc:0.79951
[50]	validation_0-auc:0.80251
[60]	validation_0-auc:0.80510
[70]	validation_0-auc:0.80674
[80]	validation_0-auc:0.80869
[90]	validation_0-auc:0.81010
[100]	validation_0-auc:0.81115
[110]	validation_0-auc:0.81202
[120]	validation_0-auc:0.81305
[130]	validation_0-auc:0.81405
[140]	validation_0-auc:0.81475
[150]	validation_0-auc:0.81580
[160]	validation_0-auc:0.81646
[170]	validation_0-auc:0.81707
[180]	validation_0-auc:0.81775
[190]	validation_0-auc:0.81829
[200]	validation_0-auc:0.81865
[210]	validation_0-auc:0.81887
[220]	validation_0-auc:0.81925
[230]	validation_0-auc:0.81965
[240]	validation_0-auc:0.81996
[250]	validation_0-auc:0.82030
[260]	validation_0-auc:0.82047
[270]	validation_0-auc:0.82097
[280]	validation_0-auc:0.82115
[290]	validation_0-auc:0.82140
[300]	validation_0-auc:0.82180
[310]	validation_0-auc:0.82209
[320]	validation_0-

In [2]:
import torch
from src.models import create_model_from_config

# Load DNN predictions
model = create_model_from_config(config)
model.load_state_dict(torch.load("models/dnn/final/model.pt"))
model.eval()
with torch.no_grad():
    dnn_proba = model(torch.FloatTensor(X_test)).squeeze().numpy()

dnn_metrics = compute_metrics(y_test, dnn_proba)
dnn_metrics["training_time_seconds"] = 1100.4  # from training log

# Comparison table
all_metrics = {
    "Logistic Regression": lr_metrics,
    "Random Forest": rf_metrics,
    "XGBoost": xgb_metrics,
    "DNN (default)": dnn_metrics
}

print_comparison_table(all_metrics)

NameError: name 'lr_metrics' is not defined

In [ ]:
save_metrics(all_metrics, "results/metrics/all_models_comparison.json")